In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [3]:
import torch
from src.config import DATASET_ROOT, POSE_DATASET_ROOT, GAMMA_POSE_DATASET_ROOT
from src.Skeleton_model.yolo_pose_tracking import save_annotated_pose_videos
from src.rwf2000 import RWF2000PoseDataset 
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from src.Skeleton_model.stgcn import STGCN
from scripts.common.get_device import get_available_device
from ultralytics import YOLO
from pathlib import Path
import torch
import numpy as np
from src.Skeleton_model.yolo_pose_tracking import pose_data_to_stgcn_tensor

In [4]:
pose_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train")
radii = compute_joint_distance_to_center_of_gravity(pose_dataset)
skeleton_graph = SkeletonGraph(radii)
device = get_available_device()
model = STGCN(adjacency=skeleton_graph.A).to(device)

/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/Skeleton_model/graph.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.radii = torch.tensor(radii, dtype=torch.float32)


Using cuda:3 with 22.28 GB free


In [5]:
from pathlib import Path
import torch
import numpy as np

def analyse_pose_database(pose_root):
    pose_files = list(Path(pose_root).rglob("*.pt"))

    empty_samples = 0
    detection_coverages = []
    tracking_coverages = []
    visible_joints_per_frame = []
    keypoint_confidences = []
    unique_track_ids_per_video = []

    for pose_path in pose_files:
        pose_data = torch.load(pose_path, weights_only=False)
        frames = pose_data["frames"]

        frames_with_detections = 0
        frames_with_tracks = 0
        total_visible_joints = 0
        track_ids = set()
        sample_has_pose = False

        for frame in frames:
            people = frame["people"]

            if len(people) > 0:
                frames_with_detections += 1
                sample_has_pose = True

            frame_has_track = False

            for person in people:
                if person["track_id"] is not None:
                    track_ids.add(person["track_id"])
                    frame_has_track = True

                confidence = person["keypoint_confidence"]
                visible = confidence > 0

                total_visible_joints += visible.sum().item()
                keypoint_confidences.extend(
                    confidence[visible].tolist()
                )

            if frame_has_track:
                frames_with_tracks += 1

        num_frames = len(frames)

        if not sample_has_pose:
            empty_samples += 1

        detection_coverages.append(
            frames_with_detections / num_frames
        )

        tracking_coverages.append(
            frames_with_tracks / num_frames
        )

        visible_joints_per_frame.append(
            total_visible_joints / num_frames
        )

        unique_track_ids_per_video.append(
            len(track_ids)
        )

    return {
        "num_samples": len(pose_files),
        "empty_samples": empty_samples,
        "mean_detection_coverage": np.mean(detection_coverages),
        "mean_tracking_coverage": np.mean(tracking_coverages),
        "mean_visible_joints_per_frame": np.mean(visible_joints_per_frame),
        "mean_keypoint_confidence": np.mean(keypoint_confidences),
        "mean_unique_track_ids_per_video": np.mean(unique_track_ids_per_video),
    }

In [6]:
original_results = analyse_pose_database(
    POSE_DATASET_ROOT
)


print("ORIGINAL")
for key, value in original_results.items():
    print(f"{key}: {value}")


ORIGINAL
num_samples: 2000
empty_samples: 288
mean_detection_coverage: 0.6915333333333334
mean_tracking_coverage: 0.6915333333333334
mean_visible_joints_per_frame: 28.619839999999996
mean_keypoint_confidence: 0.6205726204782711
mean_unique_track_ids_per_video: 6.101
